# ROBT407: Dropout for Fashion-MNIST clothing classification

**Research question:** Does adding dropout to a small CNN classification head improve macro-F1 on unseen clothing images? Which garments remain difficult?

This notebook is the experiment entry point. Source cells and the hypothesis were written before model fitting. `train` fits all models; `verify` replays saved evidence and checks every checkpoint against its saved test predictions. All ten categories are included. Team names and member contributions will be completed by the team before submission.

## 1. Prediction and fixed protocol (recorded before training)

**Hypothesis:** Head dropout at **p = 0.30** will increase mean held-out macro-F1 by reducing co-adaptation of the 64 hidden head features and the tendency of the dense head to fit training-specific details. The effect may be small: a compact CNN and a large training set can limit overfitting, while dropout can slow learning within a fixed budget. A negative or seed-dependent result remains valid.

**One change:** the dropout probability between the head ReLU and final linear layer changes from 0 to 0.30. Trainable layers and parameter counts stay identical. Both variants start from paired identical weights and use an independent shuffle generator for identical minibatch order. Fixed: split seed 407, pixel preprocessing, Adam learning rate 0.001, no weight decay or augmentation, batch size 128, 12 epochs, and highest validation macro-F1 checkpoint (earliest tie). Seeds: 11, 22, 33. No CNN hyperparameter search is conducted: one prescribed setting per variant and three training repeats. Logistic regression tunes C in {0.01, 0.1, 1} on the same validation set.

**Judgment:** primary metric is macro-F1, the unweighted mean of ten per-class F1 scores. Report all seeds, sample standard deviations (ddof=1), and each paired difference (dropout minus CNN). A positive mean is directional support in this fixed experiment; inconsistent signs weaken it. Three seeds describe training variation, not population significance. Accuracy and balanced accuracy are secondary metrics. Test metrics are computed only after every model and checkpoint is frozen. Representative CNN error plots use seed 11, chosen here before seeing results.

**Inclusion and duplicate rules:** retain every official test example. Hash raw image pixels; remove development images with a hash present in test, remove conflicting-label development groups, and keep the first original index of every remaining exact duplicate group. Create one stratified 80/20 train/validation split of the remaining training pool. No object IDs are available. Internal official test duplicates remain and are disclosed. Near duplicates are outside the exact-hash audit. Fit a single global pixel mean and standard deviation on training images only.

In [ ]:
from pathlib import Path
import os, json, pandas as pd, numpy as np
from IPython.display import display, Image
from src.study import Study, verify_saved
from src.data import LABELS
ROOT = Path.cwd()
MODE = os.environ.get('FASHION_MODE', 'train')
print('Mode:', MODE)
print(json.dumps(json.loads((ROOT/'config.json').read_text()), indent=2))
study = Study(ROOT)
display(study.hardware())

## 2. Data provenance and partition audit

Fashion-MNIST: Xiao, Rasul and Vollgraf (2017), arXiv:1708.07747v2, https://arxiv.org/abs/1708.07747. Original IDX files: https://github.com/zalandoresearch/fashion-mnist. Download with `python scripts/download_data.py`; the loader checks the authors' four MD5 values. This identifies the original release by content rather than a changing repository branch. We implement the IDX loader locally and do not reuse a tutorial training script. The training loop follows standard PyTorch practices described in https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html. Dropout rationale: Srivastava et al., JMLR 15 (2014), https://jmlr.org/papers/v15/srivastava14a.html.

The following summary records missing-data checks, label mapping, exact duplicates and every class count. Test image hashing is a contamination check, not performance-based selection.

In [ ]:
display(study.d['audit'])
display(pd.DataFrame(study.d['audit']['class_counts'], index=LABELS))
print('Training-only normalization:', study.d['mean'], study.d['std'])

## 3. Implementation and sanity checks

Input N×1×28×28; Conv(1,16,3,pad=1), ReLU, MaxPool(2); Conv(16,32,3,pad=1), ReLU, MaxPool(2); flatten 1568; Linear(1568,64), ReLU, Dropout(p), Linear(64,10). Cross-entropy consumes raw logits and int64 class targets. `train()` enables dropout; evaluation uses `eval()` and `inference_mode()`. Each minibatch clears gradients, computes logits and loss, backpropagates, then applies Adam. Checkpoints use validation only. The subset-learning check uses 64 training images and is discarded afterward.

In [ ]:
from src.study import SmallCNN
print(SmallCNN())
if MODE == 'train': display(study.checks())
else: display(json.loads((ROOT/'results/checks.json').read_text()))

## 4. Classical baseline and validation selection
Multinomial logistic regression uses flattened normalized pixels and L2 regularization. All three C candidates must converge before the protocol can freeze. A pre-test setup run hit the original 500-iteration cap for C=0.1 and C=1; the common cap was raised to 5000 and all candidates rerun. These three initial fits and their times are disclosed in results/development_log.json; the CNN protocol stayed fixed. Training uses only the training partition, without refitting on validation.

In [ ]:
if MODE == 'train': display(study.baseline())
else: display(pd.DataFrame(json.loads((ROOT/'results/baseline_tuning.json').read_text())['candidates']))

## 5. Paired CNN training
Every run receives the full 12-epoch budget. We retain the checkpoint with the largest validation macro-F1. Online training loss/F1 use changing weights and active dropout; validation uses fixed epoch-end weights and dropout off, so their gap is descriptive rather than an exact like-for-like generalization gap.

In [ ]:
if MODE == 'train': display(study.train_cnns())
else: display(pd.read_csv(ROOT/'results/training_runs.csv'))

## 6. Freeze before test evaluation
Save the chosen settings, checkpoints, split and source hashes before computing any test metric. Reusing saved predictions in verification does not select models or change settings.

In [ ]:
if MODE == 'train':
    frozen = study.freeze()
    print('Procedure frozen at', frozen['frozen_at_utc'])
    display(study.evaluate())
else:
    display(json.loads((ROOT/'results/frozen_protocol.json').read_text())['selected_cnns'])
    display(pd.read_csv(ROOT/'results/comparison.csv'))

## 7. Reproduce numbers and inspect evidence
Metrics below are recomputed from saved official-test predictions. Mean and sample SD aggregate training seeds, not independent test samples. Every model sees the same 10,000 indices.

In [ ]:
from src.analysis import build_analysis
display(verify_saved(ROOT, check_checkpoints=True))
build_analysis(ROOT)
display(pd.read_csv(ROOT/'results/summary.csv'))
display(pd.read_csv(ROOT/'results/matched_seed_deltas.csv'))

In [ ]:
import runpy
runpy.run_path(str(ROOT/'scripts/refine_figures.py'))
for figure in ['comparison','learning_curves','confusion_matrices','class_f1','representative_errors']:
    display(Image(filename=str(ROOT/f'figures/{figure}.png')))
display(pd.read_csv(ROOT/'results/error_pairs.csv').head(20))

## 8. Interpretation and limitations

The completed interpretation is grounded in the tables and images above and the report. Error examples and garment-shape explanations formed after inspecting the test set are **exploratory**. Fashion-MNIST is low-resolution grayscale catalog imagery; performance does not establish accuracy on real camera photos, new clothing sources or demographic groups. Exact deduplication cannot detect every related item. A single split, one dropout rate, one CNN size and a fixed budget limit generalization. Three seeds quantify training variability only.

No pretrained weights or external training data are used. The baseline and neural models use the same examples and normalization but have different representation families; the controlled causal comparison is CNN versus CNN plus dropout. Code was prepared with AI assistance; team members must review and understand it and supply their actual names and contributions. Recording and Moodle submission remain pending.